In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from libpysal import graph
from sklearn import cluster

In [ ]:
chicago = gpd.read_file(
    "https://martinfleischmann.net/sds/clustering/data/chicago_influenza_1918.geojson"
)
chicago.explore()

In [ ]:
from sklearn import preprocessing

demographics = [
    "gross_acres",
    "illit",
    "unemployed_pct",
    "ho_pct",
    "agecat1",
    "agecat2",
    "agecat3",
    "agecat4",
    "agecat5",
    "agecat6",
    "agecat7",
]
chicago[demographics] = preprocessing.robust_scale(chicago[demographics])
chicago.head(2)

In [ ]:
_ = sns.pairplot(chicago[demographics],height=1, plot_kws={"s":1})

In [ ]:
kmeans5 = cluster.KMeans(n_clusters=5, random_state=42)

In [ ]:
kmeans5.fit(chicago[demographics])

In [ ]:
kmeans5.labels_

In [ ]:
chicago["kmeans_5"] = kmeans5.labels_
chicago["kmeans_5"].head()

In [ ]:
chicago[["kmeans_5", 'geometry']].explore("kmeans_5", categorical=True, tiles="CartoDB Positron")

In [ ]:
inertias = {}

for k in range(2, 15):
    kmeans = cluster.KMeans(n_clusters=k, random_state=42)
    kmeans.fit(chicago[demographics])
    inertias[k] = kmeans.inertia_

In [ ]:
_ = pd.Series(inertias).plot()

In [ ]:
kmeans7 = cluster.KMeans(n_clusters=7, random_state=42)

In [ ]:
kmeans7.fit(chicago[demographics])

In [ ]:
chicago["kmeans_7"] = kmeans7.labels_
chicago["kmeans_7"].head()

In [ ]:
chicago[["kmeans_7", 'geometry']].explore("kmeans_7", categorical=True, tiles="CartoDB Positron")

fig, axes = plt.subplots(1, 7, figsize=(10, 3), sharey=True)

for i, ax in enumerate(axes):
    ax.set_title(f'Cluster {i}')
    ax.tick_params(axis='x', rotation=90)
    
    sns.boxplot(
        data=chicago.loc[chicago['kmeans_7'] == i, subranks],
        ax=ax
    )

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(10, 3), sharey=True)

for i, ax in enumerate(axes):
    ax.set_title(f'Cluster {i}')
    ax.tick_params(axis='x', rotation=90)
    
    sns.boxplot(
        data=chicago.loc[chicago['kmeans_7'] == i, demographics],
        ax=ax
    )

In [ ]:
queen = graph.Graph.build_contiguity(chicago)

In [ ]:
queen_row = queen.transform("R")

In [ ]:
for column in demographics:
    chicago[column + "_lag"] = queen_row.lag(chicago[column])

In [ ]:
chicago.info()

In [ ]:
demographics_lag = [column + "_lag" for column in demographics]
demographics_lag

In [ ]:
demographics_spatial = demographics + demographics_lag
demographics_spatial

In [ ]:
kmeans7_lag = cluster.KMeans(n_clusters=7, random_state=42)

In [ ]:
kmeans7_lag.fit(chicago[demographics_spatial])

In [ ]:
chicago["kmeans_7_lagged"] = kmeans7_lag.labels_

In [ ]:
chicago[["kmeans_7_lagged", 'geometry']].explore("kmeans_7_lagged", categorical=True, tiles="CartoDB Positron")

In [ ]:
agg5 = cluster.AgglomerativeClustering(n_clusters=5, connectivity=queen.sparse)

In [ ]:
agg5.fit(chicago[demographics])

In [ ]:
chicago["agg_5"] = agg5.labels_

In [ ]:
chicago[["agg_5", 'geometry']].explore("agg_5", categorical=True, tiles="CartoDB Positron")

In [ ]:
agg7 = cluster.AgglomerativeClustering(n_clusters=7, connectivity=queen.sparse)

In [ ]:
agg7.fit(chicago[demographics])

In [ ]:
chicago["agg_7"] = agg7.labels_

In [ ]:
chicago[["agg_7", 'geometry']].explore("agg_7", categorical=True, tiles="CartoDB Positron")

In [ ]:
chicago_regions = chicago[["agg_7", "geometry"]].dissolve("agg_7")
chicago_regions

In [ ]:
chicago_regions.reset_index().explore("agg_7", categorical=True, tiles="CartoDB Positron")

In [ ]:
agg7_spatial = cluster.AgglomerativeClustering(n_clusters=7, connectivity=queen.sparse)

In [ ]:
agg7_spatial.fit(chicago[demographics_spatial])

In [ ]:
chicago["agg_7_spatial"] = agg7_spatial.labels_

In [ ]:
chicago[["agg_7_spatial", 'geometry']].explore("agg_7_spatial", categorical=True, tiles="CartoDB Positron")

In [ ]:
chicago_spatial_regions = chicago[["agg_7_spatial", "geometry"]].dissolve("agg_7_spatial")
chicago_spatial_regions

In [ ]:
chicago_spatial_regions.reset_index().explore("agg_7_spatial", categorical=True, tiles="CartoDB Positron")

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(10, 3), sharey=True)

for i, ax in enumerate(axes):
    ax.set_title(f'Cluster {i}')
    ax.tick_params(axis='x', rotation=90)
    
    sns.boxplot(
        data=chicago.loc[chicago['agg_7_spatial'] == i, demographics],
        ax=ax
    )

In [ ]:
fig, axes = plt.subplots(1, 7, figsize=(10, 3), sharey=True)

for i, ax in enumerate(axes):
    ax.set_title(f'Cluster {i}')
    ax.tick_params(axis='x', rotation=90)
    
    sns.boxplot(
        data=chicago.loc[chicago['agg_7'] == i, demographics],
        ax=ax
    )